# IEEE 118-Bus System - Security-Constrained DC Optimal Power Flow (SC-DCOPF)

Branch-oriented SC-DCOPF following the thesis formulation. Minimize the base-case generation
cost while keeping the dispatch feasible across a set of single-element ($N-1$) contingencies
$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$.

$$\min_{P,f,\theta} \sum_{g \in \mathcal{G}} c_g\, P_{g,0}\, S_{\text{base}}$$

subject to nodal balance, generator limits, thermal limits (normal rating in the base case,
emergency rating in contingencies), the DC flow law, preventive coupling for line
contingencies, the generator contingency response, bus angle limits and the reference bus.

All input data is read from `processed/`. Per-unit on a
100 MVA base.

**Build strategy:** we start with the base case only ($\mathcal{C}=\{0\}$) to validate the
mechanics against a plain DCOPF, then switch on $\mathcal{C}_T$ and $\mathcal{C}_G$.

## 1. Imports and processed data

In [182]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [183]:
# This notebook lives in a model subfolder (e.g. project/sc-dcopf/), while the
# shared processed data lives one level up. ROOT points to the project data root so paths
# survive folder reorganizations (change here only).
# SOURCE picks which processed dataset to use: 'matpower' or 'scuc'.
ROOT   = ".."
SOURCE = "matpower"   # 'matpower' or 'scuc'
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")

base_mva = 100 MVA | slack bus = 69
generators = 19 | buses = 118 | branches = 186


## 2. Index sets and branch-oriented structures

We build the sets and lookups the formulation needs. The nodal balance (3.2) is written per
bus using the sets of **outgoing** lines $\Omega_i^+$ (branches with `from_bus == i`) and
**incoming** lines $\Omega_i^-$ (`to_bus == i`), plus the generators attached to each bus
$\mathcal{G}_i$.

In [184]:
# --- sets ---
set_N = loads["bus"].tolist()          # buses      (N)
set_G = gens["gen_id"].tolist()        # generators (G)
set_L = branches["branch_id"].tolist() # branches   (L)

# --- per-element parameters ---
par_pd     = dict(zip(loads["bus"],          loads["pd_pu"]))      # P_{d,i}
par_g_bus  = dict(zip(gens["gen_id"],        gens["bus"]))         # bus of generator g
par_pmin   = dict(zip(gens["gen_id"],        gens["pmin_pu"]))     # P_g^min
par_pmax   = dict(zip(gens["gen_id"],        gens["pmax_pu"]))     # P_g^max
par_cost   = dict(zip(gens["gen_id"],        gens["c1"]))          # c_g  ($/MWh)
par_from   = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to     = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x      = dict(zip(branches["branch_id"], branches["x_pu"]))    # x_{ij}
par_rate   = dict(zip(branches["branch_id"], branches["rate_pu"])) # F_{ij}^max

# Emergency rating F_{ij}^em, used in post-contingency states. The MATPOWER data has
# rateB = rateC = rateA, so it is set as a multiple of the normal rating.
EMERG_FACTOR = 1.15
par_rate_em  = {l: EMERG_FACTOR * par_rate[l] for l in set_L}       # F_{ij}^em

# Bus angle limits theta^min / theta^max [rad] (+-0.6 rad, as in Hedman et al.)
THETA_MAX = 0.6
THETA_MIN = -THETA_MAX

# --- subset G_i : generators connected to each bus ---
set_gens_at = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)

# --- subsets Omega_i^+ (leaving bus i) and Omega_i^- (entering bus i) ---
set_out_lines = {i: [] for i in set_N}   # Omega_i^+  (from_bus == i)
set_in_lines  = {i: [] for i in set_N}   # Omega_i^-  (to_bus == i)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

# --- integrity checks ---
assert SLACK_BUS in set_N, "Slack bus not found among buses."
assert sum(len(v) for v in set_gens_at.values()) == len(set_G)
assert sum(len(v) for v in set_out_lines.values()) == len(set_L)

print(f"|N| = {len(set_N)} buses | |G| = {len(set_G)} generators | |L| = {len(set_L)} branches")
print(f"Bus {SLACK_BUS} (slack): gens={set_gens_at[SLACK_BUS]}, "
      f"lines out={len(set_out_lines[SLACK_BUS])}, lines in={len(set_in_lines[SLACK_BUS])}")

|N| = 118 buses | |G| = 19 generators | |L| = 186 branches
Bus 69 (slack): gens=[12], lines out=3, lines in=3


## 3. Operating states $\mathcal{C}$ and availability $N1$

$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$: the base case plus one scenario per
single-element contingency. $\mathcal{C}_T$ uses **non-radial lines only** (bridges are
excluded, since their outage would island a bus); $\mathcal{C}_G$ uses the real generators.

In [185]:
INCLUDE_CT = False   # transmission contingencies (non-radial lines)
INCLUDE_CG = True    # generator contingencies

# --- run identity: model / formulation / (config comes from the switches above) ---
MODEL       = "sc-dcopf"
FORMULATION = "redispatch_only"  # redispatch_only | redispatch_with_exclusions | with_load_shedding

# The set of contingencies to drop is TIED to the formulation, so choosing the formulation
# automatically sets the right behaviour (no risk of mismatched settings).
# The excluded elements were identified by the vulnerability analysis (Section 10): line 8-5 is infeasible
# even alone; line 38-65 and the generator at bus 10 are joint conflicts (feasible
# alone, infeasible together with the other contingencies).
if FORMULATION == "redispatch_with_exclusions":
    EXCLUDE = [("line", 7),    # line 8-5    (alone)
               ("line", 95),   # line 38-65  (joint)
               ("gen", 0)]     # generator at bus 10, 505 MW (joint)
else:  # 'redispatch_only' (expect infeasibility) or 'with_load_shedding' (handled by shedding)
    EXCLUDE = []

# Eligible contingency elements (drop the excluded ones)
CT_lines = [l for l in branches.loc[~branches["is_bridge"], "branch_id"].tolist()
            if ("line", l) not in EXCLUDE]                 # non-radial, not excluded
CG_gens  = [g for g in set_G if ("gen", g) not in EXCLUDE] # real generators, not excluded

# Build the scenario list. Each scenario knows which element (if any) is out of service.
scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None,
              "desc": "base case (all in service)"}]

if INCLUDE_CT:
    for l in CT_lines:
        scenarios.append({"id": len(scenarios), "type": "line",
                          "out_line": l, "out_gen": None, "desc": f"line {l} out"})
if INCLUDE_CG:
    for g in CG_gens:
        scenarios.append({"id": len(scenarios), "type": "gen",
                          "out_line": None, "out_gen": g, "desc": f"gen {g} out"})

set_C = [s["id"] for s in scenarios]     # C : operating states
scen  = {s["id"]: s for s in scenarios}  # lookup by scenario id

# N1 availability (0 = out of service, 1 = in service)
def line_on(l, c):
    return 0 if scen[c]["out_line"] == l else 1

def gen_on(g, c):
    return 0 if scen[c]["out_gen"] == g else 1

print(f"INCLUDE_CT={INCLUDE_CT} | INCLUDE_CG={INCLUDE_CG} | excluded {len(EXCLUDE)} contingencies")
print(f"Eligible: |C_T|={len(CT_lines)} non-radial lines, |C_G|={len(CG_gens)} generators")
print(f"|C| = {len(set_C)} operating states (1 base + {len(set_C) - 1} contingencies)")

# Contingency configuration name (from the switches), and the output folder built as
# results/<formulation>/<config>/ so results never overwrite a different run.
config = {
    (False, False): "basecase",       # C = {0}
    (True,  True):  "contingencies",  # base + C_T + C_G
    (True,  False): "cont_trans",     # base + C_T (lines only)
    (False, True):  "cont_gen",       # base + C_G (generators only)
}[(INCLUDE_CT, INCLUDE_CG)]

OUT_DIR = f"results/{SOURCE}/{FORMULATION}/{config}"
print(f"MODEL={MODEL} | FORMULATION={FORMULATION} | config={config}")
print(f"output -> {OUT_DIR}")

INCLUDE_CT=False | INCLUDE_CG=True | excluded 0 contingencies
Eligible: |C_T|=177 non-radial lines, |C_G|=19 generators
|C| = 20 operating states (1 base + 19 contingencies)
MODEL=sc-dcopf | FORMULATION=redispatch_only | config=cont_gen
output -> results/matpower/redispatch_only/cont_gen


### 3.1 Participation factors $\alpha_{g,c}$

In a generator contingency the lost generation is picked up by the remaining units in
proportion to their capacity:

$$\alpha_{g,c} = \frac{P_g^{\max}}{\sum_{h \in \mathcal{G}} N1_{h,c}\, P_h^{\max}}, \qquad c \in \mathcal{C}_G$$

In [186]:
par_alpha = {}
for c in set_C:
    if scen[c]["type"] != "gen":
        continue
    cap_on = sum(gen_on(h, c) * par_pmax[h] for h in set_G)   # sum_h N1_{h,c} P_h^max
    for g in set_G:
        if gen_on(g, c) == 1:
            par_alpha[(g, c)] = par_pmax[g] / cap_on

# the participation factors of each generator contingency must sum to 1
for c in set_C:
    if scen[c]["type"] == "gen":
        assert abs(sum(par_alpha[(g, c)] for g in set_G if gen_on(g, c) == 1) - 1.0) < 1e-9
print(f"alpha defined for {sum(1 for s in scenarios if s['type'] == 'gen')} generator contingencies")

alpha defined for 19 generator contingencies


## 4. Model and decision variables

In [187]:
# create empty model
model = gp.Model("SC-DCOPF_118bus")
model.setParam("OutputFlag", 1)      # set to 1 to show the solver logs so I can debug the code
model.setParam("DualReductions", 0)  # so an infeasible model is reported as INFEASIBLE (not INF_OR_UNBD), enabling IIS

# create decision variables using addVar with free bounds to be limited by the constraints
var_p = {
    (g, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"p[{g},{c}]")
    for g in set_G for c in set_C # for each gen in set G AND each contingency in set c we create one P variable
}
var_f = {
    (l, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"f[{l},{c}]")
    for l in set_L for c in set_C # for each line in set L AND each contingency in set c we create one flow variable
}
var_theta = {
    (i, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"theta[{i},{c}]")
    for i in set_N for c in set_C # for each bus in set N AND each contingency in set c we create one theta variable
}

print(f"Created {len(var_p)} p, {len(var_f)} f, {len(var_theta)} theta variables.")

Set parameter OutputFlag to value 1
Set parameter DualReductions to value 0
Created 380 p, 3720 f, 2360 theta variables.


## 5. Objective

$$\min_{P,f,\theta} \sum_{g \in \mathcal{G}} c_g\, P_{g,0}\, S_{\text{base}}$$

Only the base-case dispatch is paid for; the contingency states only constrain it.

In [188]:
model.setObjective(
    gp.quicksum(par_cost[g] * var_p[(g, 0)] * BASE_MVA # par_cost is in $/MWh and var_p is in p.u., so we multiply by BASE_MVA to get $/h
                for g in set_G),
    GRB.MINIMIZE,
)

## 6. Constraints

Each block below is one equation of the formulation, written for every scenario `c`.

### Energy Balance Constraints

$$\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{j \in \Omega_i^+} f_{ij,c} - \sum_{j \in \Omega_i^-} f_{ji,c}$$

In [189]:
for c in set_C:
    for i in set_N:
        gen_i = gp.quicksum(var_p[(g, c)] for g in set_gens_at[i])   # sum over G_i
        out_i = gp.quicksum(var_f[(l, c)] for l in set_out_lines[i]) # Omega_i^+
        in_i  = gp.quicksum(var_f[(l, c)] for l in set_in_lines[i])  # Omega_i^-
        model.addConstr(gen_i - par_pd[i] == out_i - in_i, name=f"bal[{i},{c}]")

### Generator Limits Constraints

$$P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c}$$

When a generator is out (`gen_on == 0`) both bounds collapse to 0, forcing $P_{g,c}=0$.

In [190]:
for c in set_C:
    for g in set_G:
        avail = gen_on(g, c)
        model.addConstr(var_p[(g, c)] >= par_pmin[g] * avail, name=f"pmin[{g},{c}]")
        model.addConstr(var_p[(g, c)] <= par_pmax[g] * avail, name=f"pmax[{g},{c}]")

### Thermal Transmission Limits

$$-F_{ij}^{\max} \le f_{ij,0} \le F_{ij}^{\max}$$
$$-F_{ij}^{\mathrm{em}} \le f_{ij,c} \le F_{ij}^{\mathrm{em}}, \qquad c \in \mathcal{C}_T \cup \mathcal{C}_G$$

The base case uses the normal rating; post-contingency states use the emergency rating.

In [191]:
for c in set_C:
    rate = par_rate if c == 0 else par_rate_em   # F^max in the base case, F^em in contingencies
    for l in set_L:
        model.addConstr(var_f[(l, c)] >= -rate[l], name=f"fmin[{l},{c}]")
        model.addConstr(var_f[(l, c)] <=  rate[l], name=f"fmax[{l},{c}]")

### DC Branch Power Flow Equations

$$f_{ij,c} - \tfrac{1}{x_{ij}}(\theta_{i,c} - \theta_{j,c}) = 0 \;\; (N1_{ij,c}=1), \qquad f_{ij,c} = 0 \;\; (N1_{ij,c}=0)$$

If the line is in service we impose the DC flow law; if it is out, we force $f=0$ **without**
coupling the angles (avoids the 'ghost line' that would tie $\theta_i=\theta_j$).

In [192]:
for c in set_C:
    for l in set_L:
        i, j = par_from[l], par_to[l]
        if line_on(l, c) == 1:
            model.addConstr(
                var_f[(l, c)] == (1.0 / par_x[l]) * (var_theta[(i, c)] - var_theta[(j, c)]),
                name=f"dcflow[{l},{c}]",
            )
        else:
            model.addConstr(var_f[(l, c)] == 0.0, name=f"dcflow_off[{l},{c}]")

### Preventive Coupling Constraints

$$P_{g,c} = P_{g,0}, \qquad \forall g \in \mathcal{G},\ \forall c \in \mathcal{C}_T$$

A line outage does not change the dispatch: the base-case dispatch must already be feasible
after it.

In [193]:
for c in set_C:
    if scen[c]["type"] != "line":
        continue
    for g in set_G:
        model.addConstr(var_p[(g, c)] == var_p[(g, 0)], name=f"couple[{g},{c}]")

### Generator Contingency Response

$$P_{g,c} = P_{g,0} + \alpha_{g,c} \sum_{h \in \mathcal{G}} (1 - N1_{h,c})\, P_{h,0}, \qquad \forall c \in \mathcal{C}_G,\ \forall g \in \mathcal{G} : N1_{g,c} = 1$$

The generation lost with the failed unit is shared by the remaining units according to
$\alpha_{g,c}$. The failed unit itself is held at 0 by the generator limits.

In [194]:
for c in set_C:
    if scen[c]["type"] != "gen":
        continue
    lost = gp.quicksum((1 - gen_on(h, c)) * var_p[(h, 0)] for h in set_G)   # base-case output of the failed unit
    for g in set_G:
        if gen_on(g, c) == 1:
            model.addConstr(var_p[(g, c)] == var_p[(g, 0)] + par_alpha[(g, c)] * lost,
                            name=f"resp[{g},{c}]")

### Bus Angle Limits

$$\theta^{\min} \le \theta_{i,c} \le \theta^{\max}$$

In [195]:
for c in set_C:
    for i in set_N:
        model.addConstr(var_theta[(i, c)] >= THETA_MIN, name=f"thmin[{i},{c}]")
        model.addConstr(var_theta[(i, c)] <= THETA_MAX, name=f"thmax[{i},{c}]")

### Reference Bus Constraint

$$\theta_{\mathrm{ref},c} = 0$$

In [196]:
for c in set_C:
    model.addConstr(var_theta[(SLACK_BUS, c)] == 0.0, name=f"ref[{c}]")

print(f"Model built: {model.NumVars} variables, {model.NumConstrs} constraints.")

Model built: 0 variables, 0 constraints.


## 7. Solve

In [197]:
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
DualReductions  0

Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Optimize a model with 19362 rows, 6460 columns and 32946 nonzeros (Min)
Model fingerprint: 0x1d51fd87
Model has 19 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-03, 2e+02]
  Objective range  [1e+03, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-02, 8e+01]

Presolve time: 0.03s
Presolved: 4215 rows, 4223 columns, 15138 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.03s

Barrier statistics:
 AA' NZ     : 3.157e+04
 Factor NZ  : 1.366e+05 (roughly 5 MB of memory)
 Factor Ops : 1.661e+07 (less tha

## 8. Results

With $\mathcal{C}=\{0\}$ this is a plain DCOPF, so total generation must equal total load and
the cost is the base-case dispatch cost — a sanity check before switching on contingencies.

In [198]:
if model.status == GRB.OPTIMAL:
    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    print("=" * 55)
    print("  SC-DCOPF SOLUTION — IEEE 118-Bus")
    print("=" * 55)
    print(f"  Base-case cost  : ${model.ObjVal:,.2f} /h")
    print(f"  Total load      : {total_load:.2f} MW")
    print(f"  Total generation: {total_gen:.2f} MW (base case)")
    print(f"  Scenarios       : {len(set_C)}")
elif model.status == GRB.INFEASIBLE:
    import os, re
    print("INFEASIBLE. Computing IIS to locate the conflict...")
    model.computeIIS()

    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)

    # 1) full IIS file (Gurobi native) - the ground truth
    model.write(f"{OUT}/iis_full.ilp")

    # 2) deterministic per-scenario summary, parsed from constraint names like 'fmax[12,49]'
    #    (name pattern: <type>[..,<scenario_id>]). Only constraints flagged by the IIS are kept.
    pat = re.compile(r"^([a-zA-Z_]+)\[.*,(\d+)\]$")
    recs = []
    for con in model.getConstrs():
        if con.IISConstr:
            mm = pat.match(con.ConstrName)
            if mm:
                recs.append({"scenario_id": int(mm.group(2)), "ctype": mm.group(1)})
    iis_df = pd.DataFrame(recs)
    summary = (iis_df.groupby("scenario_id")
               .agg(n_constraints=("ctype", "size"),
                    constraint_types=("ctype", lambda s: ",".join(sorted(set(s)))))
               .reset_index())
    summary["scenario_desc"] = summary["scenario_id"].map(lambda c: scen[c]["desc"])

    # translate each conflicting scenario to the physical element that is out of service,
    # so 'scenario 49' is immediately readable as 'line 38-37' (or 'gen bus 10').
    par_label = dict(zip(branches["branch_id"], branches["label"]))
    def out_element(c):
        s = scen[c]
        if s["out_line"] is not None:
            return f"line {par_label[s['out_line']]} (branch_id {s['out_line']})"
        if s["out_gen"] is not None:
            return f"gen bus {par_g_bus[s['out_gen']]} (gen_id {s['out_gen']})"
        return "base case"
    summary["out_element"] = summary["scenario_id"].map(out_element)

    summary = summary[["scenario_id", "scenario_desc", "out_element",
                       "n_constraints", "constraint_types"]]
    summary.to_csv(f"{OUT}/iis_summary.csv", index=False)

    print(f"Saved iis_full.ilp and iis_summary.csv to {OUT}/")
    print(f"Conflicting scenarios: {summary['scenario_id'].tolist()}")
    display(summary)
else:
    print(f"Solver status: {model.status}")

INFEASIBLE. Computing IIS to locate the conflict...
Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
DualReductions  0

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s

IIS computed: 1497 constraints and 0 bounds
IIS runtime: 0.83 seconds (0.50 work units)
Saved iis_full.ilp and iis_summary.csv to results/matpower/redispatch_only/cont_gen/
Conflicting scenarios: [0, 1, 4, 12, 16, 17]


,scenario_id,scenario_desc,out_element,n_constraints,constraint_types
0,0,base case (all in service),base case,283,"bal,dcflow,fmin,pmax"
1,1,gen 0 out,gen bus 10 (gen_id 0),303,"bal,dcflow,fmin,pmax,resp,thmin"
2,4,gen 3 out,gen bus 26 (gen_id 3),300,"bal,dcflow,fmin,resp"
3,12,gen 11 out,gen bus 66 (gen_id 11),20,"pmax,resp"
4,16,gen 15 out,gen bus 89 (gen_id 15),302,"bal,dcflow,fmin,pmax,resp"
5,17,gen 16 out,gen bus 100 (gen_id 16),288,"bal,dcflow,fmin,resp"


In [199]:
# Base-case dispatch of the active generators
if model.status == GRB.OPTIMAL:
    rows = []
    for g in set_G:
        mw = var_p[(g, 0)].X * BASE_MVA
        if mw > 0.01:
            rows.append({"gen_id": g, "bus": par_g_bus[g],
                         "p_mw": round(mw, 2),
                         "pmax_mw": round(par_pmax[g] * BASE_MVA, 2),
                         "c1": round(par_cost[g], 3)})
    df_dispatch = pd.DataFrame(rows).sort_values("c1").reset_index(drop=True)
    print(f"Active generators: {len(df_dispatch)}")
    df_dispatch

## 9. Export results for documentation

Saves the base-case solution to `results/` (only when the model is optimal), so results can
be documented and reused. Three files:
- `basecase_summary.csv`     — cost, load, generation, scenario/generator counts
- `basecase_dispatch.csv`    — per-generator output and loading
- `basecase_line_loading.csv`— per-line flow and loading (sorted by loading), to spot congestion

In [200]:
import os
import math

if model.status == GRB.OPTIMAL:
    OUT = OUT_DIR
    os.makedirs(OUT, exist_ok=True)
    par_label = dict(zip(branches["branch_id"], branches["label"]))

    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    n_active   = sum(1 for g in set_G if var_p[(g, 0)].X * BASE_MVA > 0.01)

    # 1) summary
    pd.DataFrame({
        "metric": ["base_cost_usd_per_h", "total_load_mw", "total_generation_mw",
                   "n_scenarios", "n_active_generators"],
        "value":  [round(model.ObjVal, 2), round(total_load, 2), round(total_gen, 2),
                   len(set_C), n_active],
    }).to_csv(f"{OUT}/summary.csv", index=False)

    # 2) per-generator dispatch (base case, c = 0)
    disp = []
    for g in set_G:
        mw   = var_p[(g, 0)].X * BASE_MVA
        pmax = par_pmax[g] * BASE_MVA
        disp.append({"gen_id": g, "bus": par_g_bus[g], "p_mw": round(mw, 2),
                     "pmax_mw": round(pmax, 2),
                     "loading_pct": round(100 * mw / pmax, 1) if pmax > 0 else 0,
                     "c1": round(par_cost[g], 3)})
    pd.DataFrame(disp).sort_values("c1").to_csv(f"{OUT}/dispatch.csv", index=False)

    # 3) per-line loading (base case) — sorted to reveal congestion
    fl = []
    for l in set_L:
        f    = var_f[(l, 0)].X * BASE_MVA
        fmax = par_rate[l] * BASE_MVA
        fl.append({"branch_id": l, "label": par_label[l], "f_mw": round(f, 2),
                   "fmax_mw": round(fmax, 2),
                   "loading_pct": round(100 * abs(f) / fmax, 1) if fmax > 0 else 0})
    (pd.DataFrame(fl).sort_values("loading_pct", ascending=False)
        .to_csv(f"{OUT}/line_loading.csv", index=False))

    # 4) bus voltage angles (base case). Slack is fixed at 0; others are relative to it.
    ang = []
    for i in set_N:
        rad = var_theta[(i, 0)].X
        ang.append({"bus": i, "theta_rad": round(rad, 6),
                    "theta_deg": round(math.degrees(rad), 4),
                    "is_slack": i == SLACK_BUS})
    (pd.DataFrame(ang).sort_values("theta_deg")
        .to_csv(f"{OUT}/angles.csv", index=False))

    print(f"Saved 4 files to {OUT}/ (cost=${model.ObjVal:,.2f}/h, {n_active} active gens).")
else:
    print("Model not optimal - nothing exported.")

Model not optimal - nothing exported.


## 10. Vulnerability analysis (optional)

When the full model is infeasible, this section finds **which** contingencies cause it. It
reuses the model built above: to test a subset of states, it copies the model and removes
the constraints of every other state (constraint names end in `,c]`, so each one is mapped
to its scenario).

The states are coupled through the base-case dispatch $P_{g,0}$, so every test includes
the base case:
1. **Alone** - base case + one contingency. Infeasible here means infeasible in any set.
2. **Joint** - contingencies are added one at a time, in scenario order, to a growing
   feasible set; one that breaks feasibility is a joint conflict and is left out. The
   result depends on the order: a joint conflict is a conflict with the contingencies
   accepted before it.

Run it with `FORMULATION = "redispatch_only"` and both `INCLUDE_CT` / `INCLUDE_CG` on, so
every contingency is tested. The weak points are the candidates for `EXCLUDE`.

In [201]:
RUN_VULNERABILITY = False

if RUN_VULNERABILITY:
    import os, re

    # scenario of every constraint, from its name (e.g. 'fmax[12,49]' -> 49, 'ref[3]' -> 3)
    pat = re.compile(r"(\d+)\]$")
    con_scen = [int(pat.search(con.ConstrName).group(1)) for con in model.getConstrs()]

    def solve_states(keep):
        # Solve the model keeping only the states in 'keep' (the base case is always kept).
        keep = set(keep) | {0}
        m = model.copy()
        m.setParam("OutputFlag", 0)
        cons = m.getConstrs()
        m.remove([cons[k] for k, s in enumerate(con_scen) if s not in keep])
        m.optimize()
        return m.status, (m.ObjVal if m.status == GRB.OPTIMAL else None)

    par_label = dict(zip(branches["branch_id"], branches["label"]))
    def element(c):
        s = scen[c]
        if s["out_line"] is not None:
            return f"line {par_label[s['out_line']]}"
        return f"gen bus {par_g_bus[s['out_gen']]} ({par_pmax[s['out_gen']] * BASE_MVA:.0f} MW)"

    conts = [c for c in set_C if c != 0]
    vuln = pd.DataFrame({"scenario_id": conts,
                         "type": [scen[c]["type"] for c in conts],
                         "element": [element(c) for c in conts]})

    # 1) alone: base case + one contingency
    vuln["feasible_alone"] = [solve_states([c])[0] == GRB.OPTIMAL for c in conts]

    # 2) joint: add the contingencies one at a time to a growing feasible set
    accepted, joint = [], []
    for c, ok in zip(conts, vuln["feasible_alone"]):
        if ok and solve_states(accepted + [c])[0] == GRB.OPTIMAL:
            accepted.append(c)
            joint.append(True)
        else:
            joint.append(False)
    vuln["feasible_joint"] = joint

    vuln["problem"] = "none"
    vuln.loc[~vuln["feasible_alone"], "problem"] = "alone"
    vuln.loc[vuln["feasible_alone"] & ~vuln["feasible_joint"], "problem"] = "joint"
    weak = vuln[vuln["problem"] != "none"].reset_index(drop=True)

    status, cost = solve_states(accepted)
    print(f"Tested {len(conts)} contingencies | {len(weak)} weak points | "
          f"accepted set: {len(accepted)} contingencies, cost ${cost:,.2f}/h")

    OUT = f"results/{SOURCE}/vulnerability"
    os.makedirs(OUT, exist_ok=True)
    vuln.to_csv(f"{OUT}/vulnerability_full.csv", index=False)
    weak.to_csv(f"{OUT}/vulnerability_weak_points.csv", index=False)
    print(f"Saved to {OUT}/")
    display(weak)